## Ejemplo de Clasificación con Spark MLlib
En este notebook realizaremos un ejemplo completo de clasificación binaria con PySpark y MLlib.
Prediremos si un cliente abandonará (churn) una compañía de telecomunicaciones en función de sus características.
Como en el ejemplo de regresión, trabajaremos con un dataset sintético para que puedas ejecutarlo sin descargar archivos externos.

# 1. Instalación de PySpark en Colab

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import rand, when, col
import pandas as pd
import numpy as np

# Crear sesión Spark
spark = SparkSession.builder \
    .appName("ClasificacionConMLlib") \
    .getOrCreate()

print("Spark sesión iniciada correctamente")

Spark sesión iniciada correctamente


## 2. Crear dataset sintético de churn
Generamos 10 000 registros con características demográficas y de uso. La variable objetivo es churn (1 si abandona, 0 si no).

In [3]:
num_rows = 10000

data = spark.range(num_rows).select(
    (rand() * 60 + 18).cast("int").alias("edad"),            # 18-78 años
    (rand() * 5).cast("int").alias("num_llamadas_queja"),    # 0-4 quejas
    (rand() * 100).cast("int").alias("consumo_mensual_gb"),  # 0-99 GB
    (rand() * 24 + 1).cast("int").alias("antiguedad_meses"), # 1-24 meses
    when(rand() < 0.5, "residencial").otherwise("empresarial").alias("tipo_contrato"),
    when(rand() < 0.33, "Madrid")
        .when(rand() < 0.66, "Barcelona")
        .otherwise("Valencia").alias("ciudad")
)

# Regla para generar churn: combina factores
data = data.withColumn("churn",
    when(
        (col("num_llamadas_queja") >= 3) |
        (col("antiguedad_meses") <= 6) |
        ((col("consumo_mensual_gb") > 80) & (col("tipo_contrato") == "residencial")),
        1.0
    ).otherwise(0.0)
)

# Añadir ruido: 10% de los registros se etiquetan aleatoriamente
data = data.withColumn("churn",
    when(rand() < 0.1, 1 - col("churn")).otherwise(col("churn"))
)

# Mostrar ejemplos
data.show(5, truncate=False)

+----+------------------+------------------+----------------+-------------+---------+-----+
|edad|num_llamadas_queja|consumo_mensual_gb|antiguedad_meses|tipo_contrato|ciudad   |churn|
+----+------------------+------------------+----------------+-------------+---------+-----+
|25  |3                 |61                |17              |empresarial  |Barcelona|1.0  |
|65  |4                 |62                |22              |empresarial  |Barcelona|1.0  |
|34  |4                 |76                |5               |residencial  |Madrid   |1.0  |
|19  |4                 |32                |21              |empresarial  |Madrid   |1.0  |
|69  |2                 |42                |13              |residencial  |Barcelona|0.0  |
+----+------------------+------------------+----------------+-------------+---------+-----+
only showing top 5 rows


## 3. Preparación de features (pipeline de preprocesamiento)
Transformamos las columnas categóricas a numéricas con StringIndexer, luego las ensamblamos en un vector y finalmente normalizamos.

In [4]:
# Indexar variables categóricas
tipo_indexer = StringIndexer(inputCol="tipo_contrato", outputCol="tipo_index")
city_indexer = StringIndexer(inputCol="ciudad", outputCol="ciudad_index")

# Vector assembler: todas las features (numéricas + índices)
feature_cols = ["edad", "num_llamadas_queja", "consumo_mensual_gb", "antiguedad_meses", "tipo_index", "ciudad_index"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")

# Escalado opcional (recomendado para regresión logística)
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

# Pipeline de preprocesamiento
preprocessing_pipeline = Pipeline(stages=[tipo_indexer, city_indexer, assembler, scaler])

prepared_data = preprocessing_pipeline.fit(data).transform(data)

prepared_data.select("features", "churn").show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                               |churn|
+-----------------------------------------------------------------------------------------------------------------------+-----+
|[-1.2971262516840756,0.7249329592156597,0.4012753904152914,0.6557873271426514,1.0015512007342948,-1.0038758640447938]  |1.0  |
|[0.9959633313025356,1.4304638197661794,0.435764332341332,1.379167276779355,1.0015512007342948,-1.0038758640447938]     |1.0  |
|[-0.781181095512088,1.4304638197661794,0.9186095193059005,-1.0803245519854374,-0.998351356642492,0.25997389554829925]  |1.0  |
|[-1.6410896891320672,1.4304638197661794,-0.5989039254398862,1.2344912868520144,1.0015512007342948,0.25997389554829925] |1.0  |
|[1.2252722896011967,0.01940209866513982,-0.2540145061794801,0.07708336743328842,-0.998351356642492,-1.0

## 4. División en entrenamiento y prueba

In [5]:
train_data, test_data = prepared_data.randomSplit([0.8, 0.2], seed=42)
print(f"Entrenamiento: {train_data.count()} registros")
print(f"Prueba: {test_data.count()} registros")

Entrenamiento: 8064 registros
Prueba: 1936 registros


## 5. Entrenamiento del clasificador (Regresión Logística)

In [6]:
lr = LogisticRegression(featuresCol="features", labelCol="churn")

lr_model = lr.fit(train_data)

# Ver coeficientes
print("Coeficientes:", lr_model.coefficients)
print("Intercepto:", lr_model.intercept)

Coeficientes: [0.009833003980926102,1.2303914975236494,0.2965373733054846,-0.7928797988187716,-0.20619090744864493,-0.008468335222811962]
Intercepto: 0.40205374073083394


## 6. Predicciones y evaluación
Generamos predicciones y calculamos métricas: accuracy, precision, recall, AUC.

In [8]:
predictions = lr_model.transform(test_data)

# Mostrar algunas predicciones
predictions.select("churn", "prediction", "probability").show(5, truncate=False)

+-----+----------+----------------------------------------+
|churn|prediction|probability                             |
+-----+----------+----------------------------------------+
|1.0  |0.0       |[0.7350332346471821,0.26496676535281793]|
|0.0  |0.0       |[0.8975325554473221,0.10246744455267787]|
|1.0  |0.0       |[0.6657130756280716,0.3342869243719284] |
|0.0  |0.0       |[0.8722030672736866,0.12779693272631343]|
|0.0  |0.0       |[0.6119354490559544,0.38806455094404557]|
+-----+----------+----------------------------------------+
only showing top 5 rows


## Métricas de clasificación

In [9]:
# Evaluador multiclase (accuracy, precision, recall)
multi_evaluator = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction")

accuracy = multi_evaluator.setMetricName("accuracy").evaluate(predictions)
precision = multi_evaluator.setMetricName("precisionByLabel").evaluate(predictions)
recall = multi_evaluator.setMetricName("recallByLabel").evaluate(predictions)

# Evaluador binario (AUC)
binary_evaluator = BinaryClassificationEvaluator(labelCol="churn", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = binary_evaluator.evaluate(predictions)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"AUC:       {auc:.4f}")

Accuracy:  0.7722
Precision: 0.7414
Recall:    0.7246
AUC:       0.8392


## 7. Uso de Pipeline completo
Como en regresión, podemos unificar todo el flujo (preprocesamiento + clasificador) en un único Pipeline.

In [10]:
# Construir pipeline completo
full_pipeline = Pipeline(stages=[
    tipo_indexer,
    city_indexer,
    assembler,
    scaler,
    lr
])

# Usar datos originales (sin transformar)
train_orig, test_orig = data.randomSplit([0.8, 0.2], seed=42)

pipeline_model = full_pipeline.fit(train_orig)
predictions_pipe = pipeline_model.transform(test_orig)

# Evaluar igual que antes
accuracy_pipe = multi_evaluator.setMetricName("accuracy").evaluate(predictions_pipe)
auc_pipe = binary_evaluator.evaluate(predictions_pipe)

print("\n--- Resultados con Pipeline completo ---")
print(f"Accuracy: {accuracy_pipe:.4f}")
print(f"AUC:      {auc_pipe:.4f}")


--- Resultados con Pipeline completo ---
Accuracy: 0.7722
AUC:      0.8392


## 8. Guardar y cargar el modelo (opcional)

In [11]:
# Guardar pipeline entrenado
pipeline_model.write().overwrite().save("modelo_churn")

# Cargarlo
from pyspark.ml import PipelineModel
loaded_model = PipelineModel.load("modelo_churn")

## Conclusión
Hemos implementado un flujo completo de clasificación binaria con Spark MLlib en Colab, incluyendo:

* Generación de datos sintéticos con una regla de churn.

* Preprocesamiento con indexación de categóricas, ensamblado y escalado.

* Entrenamiento de regresión logística.

* Evaluación con accuracy, precision, recall y AUC.

* Uso de Pipeline para unificar el proceso.

Al igual que en regresión, no se requiere GPU – con CPU es suficiente.

Spark MLlib permite escalar este tipo de análisis a conjuntos de datos masivos distribuidos, manteniendo una sintaxis similar a scikit-learn.